# 第 15 章：SFT 指令微调

对应新版 `roadmap.md` 主线入口。

**核心问题**：怎么让模型从“续写文本”变成“按指令回答”？

**本章关注**：instruction tuning, chat dataset, assistant-only loss, JSON schema validation, 过拟合观察。

**轻量实验**：构造一条 chat 样本，并只给 assistant token 保留 loss。

> 说明：本 notebook 只做最小可运行观察，不做大型训练；如果后续要接入仓库内 API，请先确认 API 已存在。


In [ ]:
messages = [
    ("system", "你是谨慎的领域学习助手。"),
    ("user", "这条合同有没有风险？"),
    ("assistant", '{"risk_level":"unknown","needs_human_review":true}'),
]

tokens = []
labels = []
for role, content in messages:
    role_tokens = [f"<{role}>"] + content.split()
    tokens.extend(role_tokens)
    if role == "assistant":
        labels.extend(role_tokens)
    else:
        labels.extend([-100] * len(role_tokens))

print("tokens:", tokens)
print("labels:", labels)
assert any(label == -100 for label in labels)
assert any(label != -100 for label in labels)


## 学习观察

运行上面的最小实验后，建议记录三点：

1. 哪个输入或配置最影响输出？
2. 这个 toy 实验和本章核心问题之间的对应关系是什么？
3. 如果要进入 `src/` 或真实模型实现，还缺哪些已确认的 API、测试或数据？

本章验收时优先看能否解释：怎么让模型从“续写文本”变成“按指令回答”？
